# Guardrails & Human Approval

This notebook adds safety and governance controls to the Airport Operations AI Copilot.

The system will evaluate proposed operational actions before allowing them to execute.

Guardrails will check permissions, policy limits, risk level, and approval requirements.

The goal is to ensure that the AI can recommend an action without automatically performing an action that requires human authorization.

In [1]:
# Certificate issue resolve
import os

os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

import sys

# Add the project root for imports
sys.path.append("..")

from src.tools import trigger_surge_override

print("Environment ready")

Environment ready


## 1. Define Permission Levels

Different operational actions can require different levels of authorization.

For this project, a normal recommendation can be generated by the AI, while sensitive operational changes require an approval check.

This separates the ability to recommend an action from the ability to execute it.

The permission layer will be expanded with policy-based approval rules in the following steps.

In [2]:
# Define the roles used by the guardrail system
ROLE_PERMISSIONS = {
    "analyst": ["view_metrics", "generate_recommendation"],
    "operations_manager": [
        "view_metrics",
        "generate_recommendation",
        "approve_surge"
    ],
    "admin": [
        "view_metrics",
        "generate_recommendation",
        "approve_surge",
        "execute_surge"
    ]
}

print(ROLE_PERMISSIONS)

{'analyst': ['view_metrics', 'generate_recommendation'], 'operations_manager': ['view_metrics', 'generate_recommendation', 'approve_surge'], 'admin': ['view_metrics', 'generate_recommendation', 'approve_surge', 'execute_surge']}


## 2. Create a Permission Check

The permission check determines whether a user role is allowed to perform a requested action.

The AI should not assume that generating a recommendation gives it permission to execute that action.

A denied permission should stop execution and return a structured error.

This provides the first guardrail around operational actions.

In [3]:
# Check whether a role has permission for an action
def check_permission(role, action):
    # Reject unknown roles
    if role not in ROLE_PERMISSIONS:
        return {
            "status": "error",
            "message": f"Unknown role: {role}"
        }

    # Check whether the requested action is allowed
    if action not in ROLE_PERMISSIONS[role]:
        return {
            "status": "denied",
            "message": f"Role '{role}' cannot perform '{action}'"
        }

    return {
        "status": "allowed",
        "role": role,
        "action": action
    }


# Test permission checks
print(check_permission("analyst", "execute_surge"))
print(check_permission("admin", "execute_surge"))

{'status': 'denied', 'message': "Role 'analyst' cannot perform 'execute_surge'"}
{'status': 'allowed', 'role': 'admin', 'action': 'execute_surge'}


## 3. Define Surge Risk Classification

A surge change can have different levels of operational risk depending on the requested multiplier.

The risk classification provides a structured way to determine whether additional approval is required.

The SFO policy states that surge above 1.3x requires Operations Manager approval, while 1.5x is the maximum permitted multiplier.

These policy constraints will be enforced before execution.

In [4]:
# Classify the risk of a proposed surge change
def classify_surge_risk(airport_code, new_multiplier):
    # Reject unsupported airports
    if airport_code not in ["SFO", "LAX", "JFK"]:
        return {
            "status": "error",
            "message": f"Invalid airport code: {airport_code}"
        }

    # Block multipliers above the project policy maximum
    if new_multiplier > 1.5:
        return {
            "status": "blocked",
            "risk_level": "high",
            "requires_approval": False,
            "message": "Requested surge exceeds the maximum permitted limit"
        }

    # Require manager approval above 1.3x
    if new_multiplier > 1.3:
        return {
            "status": "approval_required",
            "risk_level": "high",
            "requires_approval": True,
            "message": "Operations Manager approval is required"
        }

    # Lower surge changes can proceed without manager approval
    return {
        "status": "allowed",
        "risk_level": "low",
        "requires_approval": False,
        "message": "Surge is within the standard approval range"
    }


# Test the three important policy boundaries
print("1.2x:", classify_surge_risk("SFO", 1.2))
print("1.4x:", classify_surge_risk("SFO", 1.4))
print("1.6x:", classify_surge_risk("SFO", 1.6))

1.2x: {'status': 'allowed', 'risk_level': 'low', 'requires_approval': False, 'message': 'Surge is within the standard approval range'}
1.4x: {'status': 'approval_required', 'risk_level': 'high', 'requires_approval': True, 'message': 'Operations Manager approval is required'}
1.6x: {'status': 'blocked', 'risk_level': 'high', 'requires_approval': False, 'message': 'Requested surge exceeds the maximum permitted limit'}


## 4. Policy Enforcement

Before an operational action is executed, the proposed action must be checked against the applicable airport policy.

For surge changes, the system enforces the documented maximum multiplier and approval threshold.

This prevents the AI from executing an action simply because it generated the recommendation.

In [5]:
# Validate a proposed surge action against policy limits
def enforce_surge_policy(airport_code, new_multiplier):
    # Define airport-specific maximum surge limits
    max_surge = {
        "SFO": 1.5,
        "LAX": 1.6,
        "JFK": 1.5
    }

    # Reject unsupported airports
    if airport_code not in max_surge:
        return {
            "status": "error",
            "message": f"Invalid airport code: {airport_code}"
        }

    # Block values above the airport's maximum
    if new_multiplier > max_surge[airport_code]:
        return {
            "status": "blocked",
            "message": (
                f"{airport_code} maximum surge is "
                f"{max_surge[airport_code]}x"
            )
        }

    return {
        "status": "allowed",
        "airport_code": airport_code,
        "new_multiplier": new_multiplier
    }


# Test policy enforcement
print(enforce_surge_policy("SFO", 1.4))
print(enforce_surge_policy("SFO", 1.6))

{'status': 'allowed', 'airport_code': 'SFO', 'new_multiplier': 1.4}
{'status': 'blocked', 'message': 'SFO maximum surge is 1.5x'}


## 5. Human Approval

Some actions can be technically valid but still require human authorization.

For this project, surge increases above 1.3x require Operations Manager approval.

The system therefore pauses before execution and requests explicit approval.

This creates a human-in-the-loop control instead of allowing the AI to make the final operational decision autonomously.

In [6]:
# Simulate a human approval step
def request_human_approval(
    airport_code,
    new_multiplier,
    reason,
    approved=False
):
    # Require explicit approval before sensitive execution
    if not approved:
        return {
            "status": "pending_approval",
            "airport_code": airport_code,
            "new_multiplier": new_multiplier,
            "reason": reason,
            "message": "Human approval is required before execution"
        }

    return {
        "status": "approved",
        "airport_code": airport_code,
        "new_multiplier": new_multiplier,
        "reason": reason,
        "message": "Human approval received"
    }


# Test without approval
approval_result = request_human_approval(
    "SFO",
    1.4,
    "Low completion rate and elevated queue",
    approved=False
)

print(approval_result)

{'status': 'pending_approval', 'airport_code': 'SFO', 'new_multiplier': 1.4, 'reason': 'Low completion rate and elevated queue', 'message': 'Human approval is required before execution'}


In [7]:
# Import the existing Day 2 surge execution tool
from src.tools import trigger_surge_override

print("Surge tool imported")

Surge tool imported


## 6. Guarded Surge Execution

The final execution function combines the safety checks into one controlled workflow.

The action must pass airport validation, policy enforcement, permission checks, and approval requirements before the existing surge tool is called.

The AI therefore cannot directly bypass the guardrails by calling the operational tool.

In [8]:
# Execute a surge change only after all required guardrails pass
def guarded_surge_execution(
    role,
    airport_code,
    new_multiplier,
    reason,
    approved=False
):
    # Check whether the role can execute surge changes
    permission = check_permission(role, "execute_surge")

    if permission["status"] != "allowed":
        return permission

    # Check the airport-specific policy limit
    policy = enforce_surge_policy(
        airport_code,
        new_multiplier
    )

    if policy["status"] != "allowed":
        return policy

    # Classify the action's risk
    risk = classify_surge_risk(
        airport_code,
        new_multiplier
    )

    # Request approval for actions above the approval threshold
    if risk["requires_approval"]:
        approval = request_human_approval(
            airport_code,
            new_multiplier,
            reason,
            approved
        )

        if approval["status"] != "approved":
            return approval

    # Execute the existing Day 2 tool only after checks pass
    return trigger_surge_override(
        airport_code,
        new_multiplier,
        reason
    )


# Test a blocked analyst execution
print(
    guarded_surge_execution(
        "analyst",
        "SFO",
        1.4,
        "Low completion rate"
    )
)

{'status': 'denied', 'message': "Role 'analyst' cannot perform 'execute_surge'"}


## 7. Guardrail Test Cases

The guardrail workflow should behave differently depending on the requested action.

An analyst attempting a sensitive action should be denied.
A permitted administrator action within policy should be executable.
An action requiring approval should remain pending until explicit approval is provided.
A multiplier beyond the policy maximum should always be blocked.

In [9]:
# Test a permitted low-risk execution
print("Test 1: Admin, 1.2x")
print(
    guarded_surge_execution(
        "admin",
        "SFO",
        1.2,
        "Operational supply adjustment"
    )
)

# Test an action requiring approval
print("\nTest 2: Admin, 1.4x, no approval")
print(
    guarded_surge_execution(
        "admin",
        "SFO",
        1.4,
        "Low completion rate",
        approved=False
    )
)

# Test the same action after approval
print("\nTest 3: Admin, 1.4x, approved")
print(
    guarded_surge_execution(
        "admin",
        "SFO",
        1.4,
        "Low completion rate",
        approved=True
    )
)

# Test an action above the policy maximum
print("\nTest 4: Admin, 1.6x")
print(
    guarded_surge_execution(
        "admin",
        "SFO",
        1.6,
        "Increase supply",
        approved=True
    )
)

Test 1: Admin, 1.2x
{'status': 'success', 'airport_code': 'SFO', 'new_multiplier': 1.2, 'reason': 'Operational supply adjustment', 'message': 'Surge override executed successfully (mock)'}

Test 2: Admin, 1.4x, no approval
{'status': 'pending_approval', 'airport_code': 'SFO', 'new_multiplier': 1.4, 'reason': 'Low completion rate', 'message': 'Human approval is required before execution'}

Test 3: Admin, 1.4x, approved
{'status': 'success', 'airport_code': 'SFO', 'new_multiplier': 1.4, 'reason': 'Low completion rate', 'message': 'Surge override executed successfully (mock)'}

Test 4: Admin, 1.6x
{'status': 'blocked', 'message': 'SFO maximum surge is 1.5x'}


## 8. Unified Guardrail Decision

The agent should not need to manually coordinate every safety check.

A unified guardrail function evaluates the proposed action and returns a structured decision.

The decision can be `allowed`, `pending_approval`, `denied`, or `blocked`.

This creates a single control point between the AI's recommendation and operational execution.

In [10]:
# Evaluate an operational action before execution
def evaluate_action(
    role,
    airport_code,
    action,
    new_multiplier=None,
    reason=None,
    approved=False
):
    # Check whether the requested action is supported
    if action != "execute_surge":
        return {
            "status": "blocked",
            "message": f"Unsupported action: {action}"
        }

    # Check execution permission
    permission = check_permission(role, action)

    if permission["status"] != "allowed":
        return permission

    # Validate required surge parameters
    if new_multiplier is None or reason is None:
        return {
            "status": "error",
            "message": "new_multiplier and reason are required"
        }

    # Check airport policy limits
    policy = enforce_surge_policy(
        airport_code,
        new_multiplier
    )

    if policy["status"] != "allowed":
        return policy

    # Check whether manager approval is required
    risk = classify_surge_risk(
        airport_code,
        new_multiplier
    )

    if risk["requires_approval"] and not approved:
        return {
            "status": "pending_approval",
            "risk_level": risk["risk_level"],
            "message": "Operations Manager approval required"
        }

    return {
        "status": "allowed",
        "risk_level": risk["risk_level"],
        "message": "Action passed all guardrails"
    }


# Test the unified guardrail
print(
    evaluate_action(
        "admin",
        "SFO",
        "execute_surge",
        1.2,
        "Increase supply"
    )
)

{'status': 'allowed', 'risk_level': 'low', 'message': 'Action passed all guardrails'}


## 9. Separate Decision from Execution

A key safety principle is separating the decision to execute from the actual execution.

The guardrail function decides whether an action is permitted.
Only after receiving an `allowed` decision should the operational tool be called.

This prevents a recommendation from automatically becoming an operational action.

In [11]:
# Execute an action only after the guardrail decision is allowed
def execute_guarded_action(
    role,
    airport_code,
    new_multiplier,
    reason,
    approved=False
):
    # Evaluate the action first
    decision = evaluate_action(
        role,
        airport_code,
        "execute_surge",
        new_multiplier,
        reason,
        approved
    )

    # Stop if the guardrail does not allow execution
    if decision["status"] != "allowed":
        return decision

    # Execute only after all checks pass
    result = trigger_surge_override(
        airport_code,
        new_multiplier,
        reason
    )

    return result


# Test the complete guarded execution
print(
    execute_guarded_action(
        "admin",
        "SFO",
        1.2,
        "Operational supply adjustment"
    )
)

{'status': 'success', 'airport_code': 'SFO', 'new_multiplier': 1.2, 'reason': 'Operational supply adjustment', 'message': 'Surge override executed successfully (mock)'}


## 10. Final Guardrail Validation

The final tests cover the main safety scenarios required for Day 4.

The system must distinguish between authorized actions, actions requiring approval, unauthorized actions, and actions violating policy limits.

These tests demonstrate that operational tools are protected by the guardrail layer.

In [12]:
# Define representative guardrail test cases
guardrail_tests = [
    {
        "name": "Allowed action",
        "role": "admin",
        "airport": "SFO",
        "multiplier": 1.2,
        "approved": False
    },
    {
        "name": "Approval required",
        "role": "admin",
        "airport": "SFO",
        "multiplier": 1.4,
        "approved": False
    },
    {
        "name": "Approved high-risk action",
        "role": "admin",
        "airport": "SFO",
        "multiplier": 1.4,
        "approved": True
    },
    {
        "name": "Unauthorized analyst",
        "role": "analyst",
        "airport": "SFO",
        "multiplier": 1.2,
        "approved": False
    },
    {
        "name": "Policy violation",
        "role": "admin",
        "airport": "SFO",
        "multiplier": 1.6,
        "approved": True
    }
]

for test in guardrail_tests:
    result = evaluate_action(
        test["role"],
        test["airport"],
        "execute_surge",
        test["multiplier"],
        "Operational adjustment",
        test["approved"]
    )

    print(f"{test['name']}: {result['status']}")

Allowed action: allowed
Approval required: pending_approval
Approved high-risk action: allowed
Unauthorized analyst: denied
Policy violation: blocked


In [13]:
# Import the packaged guardrail functions
from src.guardrails import (
    check_permission,
    enforce_surge_policy,
    classify_surge_risk,
    request_human_approval,
    evaluate_action,
    execute_guarded_action
)

print("Guardrail module loaded successfully")

Guardrail module loaded successfully


In [14]:
# Test the packaged execution flow
tests = [
    ("admin", "SFO", 1.2, False),
    ("admin", "SFO", 1.4, False),
    ("admin", "SFO", 1.4, True),
    ("analyst", "SFO", 1.2, False),
    ("admin", "SFO", 1.6, True)
]

for role, airport, multiplier, approved in tests:
    result = execute_guarded_action(
        role,
        airport,
        multiplier,
        "Operational supply adjustment",
        approved
    )

    print(
        role,
        airport,
        f"{multiplier}x",
        "approved=" + str(approved),
        "→",
        result["status"]
    )

admin SFO 1.2x approved=False → success
admin SFO 1.4x approved=False → pending_approval
admin SFO 1.4x approved=True → success
analyst SFO 1.2x approved=False → denied
admin SFO 1.6x approved=True → blocked


## 11. End-to-End Guardrail Demonstration

This demonstration connects the Day 3 agent workflow with the Day 4 guardrail layer.

The agent investigates the airport and generates a recommendation.
A proposed surge action is then evaluated before the operational tool is allowed to execute.

The workflow demonstrates both a blocked action and an action that proceeds after explicit approval.

In [22]:
# Certificate issue resolve
import os

os.environ['REQUESTS_CA_BUNDLE'] = '/etc/ssl/certs/ca-certificates.crt'
os.environ['SSL_CERT_FILE'] = '/etc/ssl/certs/ca-certificates.crt'

# Import required libraries
from dotenv import load_dotenv
from google import genai

# Import the packaged agent workflow
from src.agents import run_agent_loop

# Import the Day 4 guarded execution
from src.guardrails import execute_guarded_action

# Load Gemini API key
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

# Create Gemini client
client = genai.Client(api_key=api_key)

# Run the agent workflow
agent_result = run_agent_loop("SFO", client)

print("Agent workflow:", agent_result["status"])

if agent_result["status"] == "success":
    investigation = agent_result["state"]["investigation"]

    print("\nAirport:", investigation["airport_code"])
    print("Completion rate:", investigation["metrics"]["completion_rate"])
    print("Current surge:", investigation["metrics"]["surge_multiplier"])
    print("Severity:", investigation["severity"])

    # Execute guarded surge action
    guarded_result = execute_guarded_action(
        role="admin",
        airport_code="SFO",
        new_multiplier=1.4,
        reason="Improve supply during below-normal completion rate",
        approved=True
    )

    print("\nGuarded action:", guarded_result["status"])
    print("Message:", guarded_result["message"])

Agent workflow: success

Airport: SFO
Completion rate: 0.86
Current surge: 1.1
Severity: medium

Guarded action: success
Message: Surge override executed successfully (mock)


In [23]:
# Test 1: Analyst tries to execute a surge
analyst_result = execute_guarded_action(
    role="analyst",
    airport_code="SFO",
    new_multiplier=1.2,
    reason="Test analyst permission",
    approved=True
)

print("Test 1 - Analyst execution")
print("Status:", analyst_result["status"])
print("Message:", analyst_result["message"])


# Test 2: Admin requests high-risk surge without approval
pending_result = execute_guarded_action(
    role="admin",
    airport_code="SFO",
    new_multiplier=1.4,
    reason="Increase supply during operational disruption",
    approved=False
)

print("\nTest 2 - Admin without approval")
print("Status:", pending_result["status"])
print("Message:", pending_result["message"])


# Test 3: Admin requests high-risk surge with approval
approved_result = execute_guarded_action(
    role="admin",
    airport_code="SFO",
    new_multiplier=1.4,
    reason="Increase supply during operational disruption",
    approved=True
)

print("\nTest 3 - Admin with approval")
print("Status:", approved_result["status"])
print("Message:", approved_result["message"])


# Test 4: Admin requests surge above SFO policy limit
blocked_result = execute_guarded_action(
    role="admin",
    airport_code="SFO",
    new_multiplier=1.6,
    reason="Test policy limit",
    approved=True
)

print("\nTest 4 - Above policy limit")
print("Status:", blocked_result["status"])
print("Message:", blocked_result["message"])

Test 1 - Analyst execution
Status: denied
Message: Role 'analyst' cannot perform 'execute_surge'

Test 2 - Admin without approval
Status: pending_approval
Message: Operations Manager approval required

Test 3 - Admin with approval
Status: success
Message: Surge override executed successfully (mock)

Test 4 - Above policy limit
Status: blocked
Message: SFO maximum surge is 1.5x


In [24]:
# Import all packaged project components
from src.tools import get_airport_metrics
from src.agents import run_agent_loop
from src.guardrails import (
    check_permission,
    enforce_surge_policy,
    classify_surge_risk,
    execute_guarded_action
)

print("All packaged imports successful.")

All packaged imports successful.


In [25]:
# Run the complete agent workflow
agent_result = run_agent_loop("SFO", client)

print("Agent status:", agent_result["status"])

if agent_result["status"] == "success":
    investigation = agent_result["state"]["investigation"]

    print("\n--- Investigation ---")
    print("Airport:", investigation["airport_code"])
    print("Severity:", investigation["severity"])
    print("Completion rate:", investigation["metrics"]["completion_rate"])
    print("Current surge:", investigation["metrics"]["surge_multiplier"])

    print("\n--- Recommendation ---")
    print(agent_result["state"]["recommendation"])

    # Test guarded execution after investigation
    action_result = execute_guarded_action(
        role="admin",
        airport_code="SFO",
        new_multiplier=1.4,
        reason="Improve supply during operational disruption",
        approved=True
    )

    print("\n--- Guarded Action ---")
    print("Status:", action_result["status"])
    print("Message:", action_result["message"])

Agent status: success

--- Investigation ---
Airport: SFO
Severity: medium
Completion rate: 0.86
Current surge: 1.1

--- Recommendation ---
**1. Issue**
Below-normal completion rate at SFO airport (current completion rate is 86%, which is near the threshold, but the investigation flagged it as a medium-severity issue due to operational degradation relative to expected performance). *Correction based on metrics:* The current completion rate is actually 86%, but the policy states the expected rate is *above* 85%. Wait, 86% is technically above 85%, but the operational investigation explicitly flagged a "Below-normal completion rate" with medium severity. Let's look closely at the metrics: completion rate is 0.86 (86%), driver cancellation rate is 10% (0.1), queue size is 100, active drivers are 500, average ETA is 12 minutes, and surge is 1.1x. 

**2. Evidence**
* Airport: SFO
* Completion Rate: 86% (0.86)
* Driver Cancellation Rate: 10% (0.1)
* Average ETA: 12 minutes
* Active Drivers: 